# Лекция 06. Основы нейросетей / Lecture 06. Neural Network Basics

Практика: NumPy → backpropagation → PyTorch. / Practice: NumPy → backpropagation → PyTorch.

После каждой темы выполните мини-задание на 5–10 минут, сохраните вывод и пояснение. / Complete the 5–10 minute mini-task after every topic; retain output and explanation.

**Данные / Data.** UCI Optical Recognition of Handwritten Digits (E. Alpaydin, C. Kaynak, 1998), DOI [10.24432/C50P49](https://doi.org/10.24432/C50P49), [UCI](https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits), [loader documentation](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html). UCI lists CC BY 4.0; retain attribution. `load_digits` bundles 1,797 real 8×8 observations copied from the original UCI test portion. One observation = one handwritten digit; 64 block-count features in 0…16; target = digit 0…9. / UCI указывает CC BY 4.0; сохраняйте атрибуцию. 1 797 реальных изображений, одно наблюдение — цифра; 64 признака 0…16, цель — класс 0…9.

**Протокол / Protocol.** Fixed stratified teaching split: 1,078 train / 359 validation / 360 test, seed 606. Divide by known range 16; no fitted preprocessing on test. This is NOT the original benchmark split. Writer IDs are unavailable: no writer-disjoint generalization claim. Never tune using test. / Фиксированное стратифицированное учебное разбиение; это НЕ официальный benchmark. Нет идентификаторов авторов: независимость почерков не гарантируется. Test не используется для выбора модели.

**Среда / Environment.** Google Colab CPU; NumPy, scikit-learn, matplotlib, PyTorch. Tested with NumPy 2.1.3 and torch 2.8.0; data bundled with sklearn, no dataset download. / Данные входят в sklearn, скачивание датасета не требуется.

## 1. Данные и разбиение / Data and split

Покажите реальные изображения, проверьте формы. / Inspect real observations and shapes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay
import copy, json
np.set_printoptions(precision=5, suppress=True)
torch.set_num_threads(1)
torch.manual_seed(606)
print('NumPy', np.__version__, 'PyTorch', torch.__version__)
def data_split():
 d=load_digits();X=d.data.astype(np.float64)/16.;y=d.target
 tr,rest=train_test_split(np.arange(len(y)),test_size=.4,stratify=y,random_state=606)
 va,te=train_test_split(rest,test_size=.5,stratify=y[rest],random_state=606)
 assert not(set(tr)&set(va) or set(tr)&set(te) or set(va)&set(te))
 return X,y,tr,va,te
X,y,tr,va,te=data_split()
print(X.shape, len(tr), len(va), len(te))
fig,axes=plt.subplots(2,5,figsize=(10,4))
for k,ax in enumerate(axes.flat):
 i=np.flatnonzero(y==k)[0];ax.imshow(X[i].reshape(8,8),cmap='gray');ax.set_title(str(k));ax.axis('off')
plt.show()

### Мини-задание / Mini-task · 5–10 min

RU: Выведите число объектов каждого класса в train/validation/test и докажите отсутствие пересечения индексов. Объясните, почему это не доказывает независимость авторов.

EN: Count each class in train/validation/test and assert disjoint index sets. Explain why this does not establish writer independence.

In [ ]:
# Ваш код и вывод / Your code and conclusion

## 2. Single layer и MLP / Single layer and MLP

Формы: X[B,64] → H[B,32] → Z[B,10]. / Shapes: X[B,64] → H[B,32] → Z[B,10].

In [ ]:
def init_params(width=32,seed=606):
 r=np.random.default_rng(seed)
 return dict(W1=r.normal(0,np.sqrt(2/64),(64,width)),b1=np.zeros(width),W2=r.normal(0,np.sqrt(2/width),(width,10)),b2=np.zeros(10))

def forward(X,p):
 a=X@p['W1']+p['b1'];h=np.maximum(a,0);z=h@p['W2']+p['b2']
 return z,(X,a,h)
p=init_params();z,cache=forward(X[tr[:8]],p)
print('logits',z.shape,'parameters',sum(v.size for v in p.values()))
assert z.shape==(8,10)

### Мини-задание / Mini-task · 5–10 min

RU: Для ширины 16 и 64 посчитайте параметры формулой и кодом. Уберите ReLU и выведите эквивалентные W и b двух слоёв.

EN: Count parameters for widths 16 and 64 analytically and in code. Remove ReLU and derive the equivalent W and b of the two layers.

In [ ]:
# Ваш код и вывод / Your code and conclusion

## 3. Loss function / Функция потерь

Устойчивый softmax-cross-entropy, градиент среднего loss по logits. / Stable softmax cross-entropy, gradient of mean loss with respect to logits.

In [ ]:
def loss_and_grad(z,y):
 shifted=z-z.max(axis=1,keepdims=True);e=np.exp(shifted);s=e.sum(axis=1,keepdims=True);p=e/s
 loss=np.mean(np.log(s[:,0])-shifted[np.arange(len(y)),y]);g=p.copy();g[np.arange(len(y)),y]-=1
 return float(loss),g/len(y)
loss,g=loss_and_grad(z,y[tr[:8]])
print('loss',loss,'gradient shape',g.shape)
assert np.allclose(g.sum(axis=1),0)
assert np.allclose(loss_and_grad(z+10000,y[tr[:8]])[0],loss)

### Мини-задание / Mini-task · 5–10 min

RU: Прибавьте к каждой строке logits разную константу и сравните loss и градиенты. Почему argmax и loss реагируют по-разному на умножение logits на 10?

EN: Add a different constant to each row and compare loss and gradients. Why do argmax and loss respond differently to multiplying logits by 10?

In [ ]:
# Ваш код и вывод / Your code and conclusion

## 4. Backpropagation и SGD / Backpropagation and SGD

Сначала все градиенты, затем обновление. / Compute every gradient before updating.

In [ ]:
def backward(g,cache,p):
 X,a,h=cache;da=(g@p['W2'].T)*(a>0)
 return dict(W1=X.T@da,b1=da.sum(axis=0),W2=h.T@g,b2=g.sum(axis=0))

def sgd_step(p,g,lr):
 return {k:p[k]-lr*g[k] for k in p}
grads=backward(g,cache,p)
p_next=sgd_step(p,grads,.01)
print('before',loss,'after',loss_and_grad(forward(X[tr[:8]],p_next)[0],y[tr[:8]])[0])
for k in p:assert grads[k].shape==p[k].shape

### Мини-задание / Mini-task · 5–10 min

RU: Продублируйте один и тот же мини-батч дважды. Убедитесь, что средний loss и градиенты не изменились. Объясните, какую ошибку это обнаруживает.

EN: Duplicate the mini-batch and verify that mean loss and gradients remain unchanged. Explain the bug this detects.

In [ ]:
# Ваш код и вывод / Your code and conclusion

## 5. Проверяем градиенты / Verify gradients

Центральная разность — локальная проверка. Избегайте изломов ReLU. / Central differences are local diagnostics; avoid ReLU kinks.

In [ ]:
xb,yb=X[tr[:7]],y[tr[:7]]
pcheck=init_params(16)
z,cache=forward(xb,pcheck);loss,g=loss_and_grad(z,yb);analytic=backward(g,cache,pcheck)
eps=1e-5;checks=[]
for key in pcheck:
 for index in list(np.ndindex(pcheck[key].shape))[:6]:
  plus={k:v.copy() for k,v in pcheck.items()};minus={k:v.copy() for k,v in pcheck.items()}
  plus[key][index]+=eps;minus[key][index]-=eps
  numerical=(loss_and_grad(forward(xb,plus)[0],yb)[0]-loss_and_grad(forward(xb,minus)[0],yb)[0])/(2*eps)
  expected=analytic[key][index];checks.append(abs(numerical-expected))
  assert np.isclose(numerical,expected,atol=1e-7,rtol=1e-4)
print('maximum absolute discrepancy',max(checks),'min |preactivation|',np.abs(cache[1]).min())

### Мини-задание / Mini-task · 5–10 min

RU: Проверьте ещё 6 координат каждого параметра; сравните ε=10⁻³,10⁻⁵,10⁻⁷. Опишите эффект округления и пересечения излома.

EN: Check six additional coordinates per parameter; compare epsilon 10⁻³,10⁻⁵,10⁻⁷. Discuss rounding and crossing a ReLU kink.

In [ ]:
# Ваш код и вывод / Your code and conclusion

## 6. PyTorch autograd / Автоматическое дифференцирование

Сравниваем одинаковые параметры и одинаковые данные. / Compare identical parameters and data.

In [ ]:
tp={k:torch.tensor(v,dtype=torch.float64,requires_grad=True) for k,v in pcheck.items()}
tx=torch.tensor(xb,dtype=torch.float64);ty=torch.tensor(yb,dtype=torch.long)
tz=torch.relu(tx@tp['W1']+tp['b1'])@tp['W2']+tp['b2']
tl=torch.nn.functional.cross_entropy(tz,ty);tl.backward()
assert np.allclose(tl.item(),loss)
for k in tp:assert np.allclose(tp[k].grad.numpy(),analytic[k],atol=1e-10)
print('NumPy and autograd agree for loss and every gradient')

### Мини-задание / Mini-task · 5–10 min

RU: Скопируйте pcheck в nn.Sequential(Linear, ReLU, Linear), транспонируя W. Сравните logits. Выполните backward дважды и объясните накопление градиентов.

EN: Copy pcheck into nn.Sequential(Linear, ReLU, Linear), transposing W. Compare logits. Run backward twice and explain gradient accumulation.

In [ ]:
# Ваш код и вывод / Your code and conclusion

## 7. Train / eval / no_grad

Выбираем только по validation; сохраняем копию checkpoint. / Select only on validation; copy the checkpoint.

In [ ]:
def fit_torch(width,lr,epochs=45):
 torch.manual_seed(606)
 model=torch.nn.Sequential(torch.nn.Linear(64,width),torch.nn.ReLU(),torch.nn.Linear(width,10))
 opt=torch.optim.SGD(model.parameters(),lr=lr)
 tx=torch.tensor(X,dtype=torch.float32);ty=torch.tensor(y,dtype=torch.long)
 rng=np.random.default_rng(606);best_loss=float('inf');history=[]
 for epoch in range(epochs):
  model.train()
  for ids in np.array_split(rng.permutation(tr),17):
   opt.zero_grad(set_to_none=True)
   loss=torch.nn.functional.cross_entropy(model(tx[ids]),ty[ids]);loss.backward();opt.step()
  model.eval()
  with torch.no_grad():
   tl=torch.nn.functional.cross_entropy(model(tx[tr]),ty[tr]).item()
   vl=torch.nn.functional.cross_entropy(model(tx[va]),ty[va]).item()
  history.append((tl,vl))
  if vl<best_loss:best_loss=vl;best=copy.deepcopy(model.state_dict());best_epoch=epoch+1
 model.load_state_dict(best);model.eval()
 return model,{'width':width,'lr':lr,'val_loss':best_loss,'epoch':best_epoch,'history':history}
# Decide candidates before opening test / Задаём кандидатов до просмотра test.
candidates=[fit_torch(16,.1),fit_torch(32,.1)]
chosen=min(range(len(candidates)),key=lambda i:candidates[i][1]['val_loss'])
model,report=candidates[chosen]
for i,(_,r) in enumerate(candidates):print(i,{k:v for k,v in r.items() if k!='history'})
h=np.array(report['history']);plt.plot(h[:,0],label='train');plt.plot(h[:,1],label='validation');plt.xlabel('epoch');plt.ylabel('loss');plt.legend();plt.show()

### Мини-задание / Mini-task · 5–10 min

RU: До открытия test добавьте одну заранее выбранную скорость обучения. Запишите гипотезу, сравните по validation. Проверьте, что сохранённые веса не меняются при следующих шагах.

EN: Before opening test, add one predeclared learning rate. State a hypothesis and compare on validation. Verify that later steps do not mutate the saved checkpoint.

In [ ]:
# Ваш код и вывод / Your code and conclusion

## 8. Анализ ошибок / Error analysis

Test открываем после выбора. / Open test after selection.

In [ ]:
with torch.no_grad():
 predictions=model(torch.tensor(X[te],dtype=torch.float32)).argmax(1).numpy()
print('test accuracy',float(np.mean(predictions==y[te])))
ConfusionMatrixDisplay.from_predictions(y[te],predictions);plt.show()
wrong=np.flatnonzero(predictions!=y[te])[:6]
fig,axes=plt.subplots(1,6,figsize=(12,3))
for ax,j in zip(axes,wrong):
 ax.imshow(X[te[j]].reshape(8,8),cmap='gray');ax.set_title(f'{y[te[j]]} → {predictions[j]}');ax.axis('off')
for ax in axes[len(wrong):]:ax.axis('off')
plt.show()

### Мини-задание / Mini-task · 5–10 min

RU: Объясните 3 ошибки на конкретных изображениях. Укажите ограничения выборки и почему новые авторы не проверены. После просмотра test не перенастраивайте модель.

EN: Explain three errors using the actual images. State sampling limitations and why unseen writers were not evaluated. Do not retune after inspecting test.

In [ ]:
# Ваш код и вывод / Your code and conclusion

## Источники и обновление / Sources and update

Исходная последовательность / Original sequence: Single layer, Loss function, Learn, Backpropogation, NumPy/Keras, ReLU/Sigmoid (`Archive/NN_withNumpyAndKeras.ipynb`). Обновление / Update: real Digits, PyTorch, stable multiclass loss, gradient checks, validation and explicit split limitations. Old synthetic examples are not reused.

[PyTorch autograd](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html), [CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html).